In [1]:
import pandas as pd

In [2]:
# Input paths and load .csv files
grad_path = '../data/processed/edfacts_lea_graduation_2017_2018_v2.csv'
abs_path = '../data/processed/crdc_lea_absenteeism_2017_18_v2.csv'

grad_df = pd.read_csv(grad_path, low_memory=False)
abs_df = pd.read_csv(abs_path, low_memory=False)

print("Absenteeism LEAs:", abs_df['LEAID'].nunique())
print("Graduation LEAs:", grad_df['LEAID'].nunique())

Absenteeism LEAs: 15810
Graduation LEAs: 6113


In [3]:
overlap_count = abs_df['LEAID'].isin(grad_df['LEAID']).sum()
no_match_count = (~grad_df['LEAID'].isin(abs_df['LEAID'])).sum()
abs_dupes = abs_df['LEAID'].duplicated().any()
grad_dupes = grad_df['LEAID'].duplicated().any()

print(f"Overlap: {overlap_count} / {len(grad_df)} grad_df districts found in abs_df")
print(f"No match: {no_match_count}")
print(f"abs_df duplicate LEAIDs: {abs_dupes}, grad_df duplicate LEAIDs: {grad_dupes}")

Overlap: 5940 / 6113 grad_df districts found in abs_df
No match: 173
abs_df duplicate LEAIDs: False, grad_df duplicate LEAIDs: False


- `grad_df`: 6,113 distinct LEAIDs. `abs_df`: 15,810 distinct LEAIDs.
- 5,940 districts appear in both `abs_df` and `grad_df`, out of the 6,113 total districts in `grad_df`.
- 173 grad_df districts have no match in abs_df.
- Both LEAID columns confirmed duplicate-free, so this count reflects true overlap, not row-inflation.

In [4]:
mask = ~grad_df['LEAID'].isin(abs_df['LEAID'])
no_match_rows = grad_df[mask].copy()

print("Grad rate — unmatched districts:")
print(no_match_rows['grad_rate'].describe())
print("\nGrad rate — full population:")
print(grad_df['grad_rate'].describe())

Grad rate — unmatched districts:
count    173.000000
mean      52.066474
std       30.415771
min        1.000000
25%       25.000000
50%       54.000000
75%       80.000000
max       96.000000
Name: grad_rate, dtype: float64

Grad rate — full population:
count    6113.000000
mean       87.097743
std        12.595537
min         1.000000
25%        85.000000
50%        91.000000
75%        95.000000
max        99.000000
Name: grad_rate, dtype: float64


- Unmatched districts: mean grad_rate = 52.1%, std = 30.4%
- Full population: mean grad_rate = 87.1%, std = 12.6%
- The districts missing tend to be the low-performing ones, not just a random cross-section.
- The difference in std (30.4% for unmatched and 12.6% for matched districts) shows that there is large variance within the unmatched group.

In [5]:
no_match_rows['first_two'] = no_match_rows['LEAID'].astype(str).str.slice(0, 2)
no_match_rows['first_two'].value_counts()

first_two
36    34
40    24
50    24
39    17
26     9
29     9
48     8
59     8
18     7
27     5
13     4
37     3
25     3
45     3
53     2
22     2
16     2
35     1
23     1
90     1
41     1
47     1
69     1
63     1
62     1
28     1
Name: count, dtype: int64

- Top 3 state-FIPS prefixes: 36, 40, and 50 (states: NY=36, OK=40, VT=50)
- These account for 82 of the 173 unmatched districts (47%)
- This pattern is best described as Missing Not At Random (MNAR) because they skew toward much lower and more variable grad rates.
- **Join decision:** Inner join on LEAID, retaining 5,940 of 6,113 grad_df districts (97%)
- **Why not impute:** Decided against creating an artifical `absent_rate` value for these districts because it would understate absenteeism for  the districts most likely to have high absenteeism.

In [6]:
merged_df = pd.merge(grad_df, abs_df, on='LEAID', how='inner')

print(merged_df.shape)

(5940, 5)


In [7]:
# Check data after inner join

# Preview first 5 rows of merged dataframe
print(merged_df.head())

# Summary statistics for numeric columns
print(merged_df.describe())

# Count of missing (NaN) values per column
print(merged_df.isna().sum())

    LEAID  grad_rate  chron_absent  enrolled  absent_rate
0  100005       94.0         731.0    5444.0    13.427627
1  100006       91.0         928.0    5698.0    16.286416
2  100007       94.0        1330.0   14369.0     9.256037
3  100008       96.0        1066.0   10804.0     9.866716
4  100011       95.0         241.0    2027.0    11.889492
              LEAID    grad_rate  chron_absent       enrolled  absent_rate
count  5.940000e+03  5940.000000   5940.000000    5940.000000  5940.000000
mean   3.003037e+06    88.118013   1117.592761    6947.503535    15.554403
std    1.488087e+06     9.984381   3193.419220   17200.670213     9.852640
min    1.000050e+05     1.000000      0.000000      23.000000     0.000000
25%    1.807890e+06    86.000000    221.000000    1860.750000     9.297211
50%    3.302485e+06    92.000000    420.000000    3081.500000    13.671268
75%    4.203585e+06    95.000000    931.250000    5926.750000    19.218933
max    7.200030e+06    99.000000  76364.000000  5111

In [8]:
# Save processed .csv file
output_path = '../data/processed/lea_absenteeism_x_graduation_v2.csv'
merged_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

Saved to ../data/processed/lea_absenteeism_x_graduation_v2.csv
